In [1]:
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

print("Spinning up local SQL Data Warehouse...")

# Step 1: Connect to a local SQLite database
conn = sqlite3.connect('aegis_warehouse.db')

# Step 2: Load the cleaned csv data
df_clean = pd.read_csv('clean_aegis_payroll_2023.csv')

# Step 3: Push the dataframe into the SQL database as a table named 'payroll_fact'
df_clean.to_sql('payroll_fact', conn, if_exists='replace', index=False, chunksize=10000)

print("Success! Clean data loaded into 'payroll_fact' table.")

Spinning up local SQL Data Warehouse...
Success! Clean data loaded into 'payroll_fact' table.


In [6]:
# Which department is structurally burning the most overtime cash?

query_1 = """
SELECT 
    "Agency Name_Clean" AS Department,
    COUNT(*) AS Employee_Count,
    SUM("OT Hours") AS Total_OT_Hours,
    SUM("Total OT Paid") AS Total_OT_Spend
FROM payroll_fact
WHERE "OT Hours" > 0  -- Only look at employees who actually worked overtime
GROUP BY "Agency Name_Clean"
ORDER BY Total_OT_Spend DESC
LIMIT 10;
"""

# Execute the SQL query and load the results into a dataframe
dept_burnout = pd.read_sql_query(query_1, conn)

# Format the numbers so they are easy to read (adding commas and dollar signs)
dept_burnout['Total_OT_Hours'] = dept_burnout['Total_OT_Hours'].apply(lambda x: f"{x:,.2f}")
dept_burnout['Total_OT_Spend'] = dept_burnout['Total_OT_Spend'].apply(lambda x: f"${x:,.2f}")

display(dept_burnout)

,Department,Employee_Count,Total_OT_Hours,Total_OT_Spend
0,Police Department,45970,"13,618,702.47","$866,680,670.72"
1,Fire Department,16660,"7,336,071.96","$501,571,473.80"
2,Department Of Correction,7446,"4,075,407.57","$294,366,219.00"
3,Nyc Housing Authority,10328,"3,440,906.75","$186,486,356.86"
4,Department Of Sanitation,9499,"2,293,387.83","$176,452,742.45"
5,Department Of Social Services,7990,"2,682,275.23","$122,955,424.07"
6,Department Of Transportation,4656,"1,194,001.31","$78,998,315.49"
7,Dept Of Environment Protection,4117,"980,875.09","$63,548,575.49"
8,Admin For Children'S Svcs,4404,"1,066,219.47","$53,422,010.56"
9,Department Of Education,7143,"511,692.91","$32,812,094.22"


In [11]:
# Which specific roles within the Department of Correction are driving the burnout?

query_2 = """
SELECT
    "Title Description" AS Job_Role,
    COUNT(*) AS Employee_Count,
    SUM("OT Hours") AS Total_OT_Hours,
    SUM("Total OT Paid") AS Total_OT_Spend,
    (SUM("OT Hours") / COUNT(*)) AS Avg_OT_Per_Employee
FROM payroll_fact
WHERE "Agency Name_Clean" = 'Department Of Correction' AND "OT Hours" > 0
GROUP BY "Title Description"
ORDER BY Total_OT_Spend DESC
LIMIT 10;
"""

# Execute the SQL Query
role_burnout = pd.read_sql_query(query_2, conn)

# Format the numbers so they are easy to read (adding commas and dollar signs)
role_burnout['Total_OT_Hours'] = role_burnout['Total_OT_Hours'].apply(lambda x: f"{x:,.2f}")
role_burnout['Total_OT_Spend'] = role_burnout['Total_OT_Spend'].apply(lambda x: f"${x:,.2f}")
role_burnout['Avg_OT_Per_Employee'] = role_burnout['Avg_OT_Per_Employee'].apply(lambda x: f"{x:,.2f}")

display(role_burnout)

,Job_Role,Employee_Count,Total_OT_Hours,Total_OT_Spend,Avg_OT_Per_Employee
0,CORRECTION OFFICER,5811,"3,275,940.73","$226,463,497.48",563.75
1,CAPTAIN,592,"395,442.91","$35,918,362.85",667.98
2,WARDEN-ASSISTANT DEPUTY WARDEN TED < 11/1/92,93,"73,149.85","$7,287,670.74",786.56
3,ELECTRICIAN,35,"25,130.25","$2,360,911.28",718.01
4,STATIONARY ENGINEER,22,"22,584.25","$2,054,672.04","1,026.56"
5,OILER,30,"18,804.75","$1,709,114.46",626.83
6,PLUMBER,26,"12,810.75","$1,692,938.21",492.72
7,PROGRAM SPECIALIST CORRECTION,52,"20,297.25","$1,248,718.62",390.33
8,SENIOR STATIONARY ENGINEER,9,"11,817.00","$1,223,108.07","1,313.00"
9,CARPENTER,13,"12,748.00","$1,017,688.17",980.62
